## Ingest races.csv


### Step 0 - Parameters and Imports


In [0]:
%run "../utils/config"

In [0]:
dbutils.widgets.text("p_source_value", "")
v_source_value = dbutils.widgets.get("p_source_value")

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType, TimestampType, FloatType
from pyspark.sql.functions import col, lit, current_timestamp, to_timestamp, concat

### Step 1 - Read the csv file


#### 1.1 Define the schema


In [0]:
races_schema = StructType(
    [
        StructField("raceId", IntegerType(), False),
        StructField("year", IntegerType(), True),
        StructField("round", IntegerType(), True),
        StructField("circuitId", IntegerType(), True),
        StructField("name", StringType(), True),
        StructField("date", DateType(), True),
        StructField("time", StringType(), True),
        StructField("url", StringType(), True),
    ]
)


#### 1.2 Read the csv file


In [0]:
raw_path = raw_folder_path


In [0]:
races_df = (
    spark.read
        .option("header", True)
        .schema(races_schema)
        .csv(f"{raw_path}/races.csv")
)


### Step 2 - Transform the data


In [0]:
transformed_circuits_df = add_ingestion_date(
    races_df
        .withColumn(
            "race_timestamp",
            to_timestamp(
                concat(col("date"), lit(" "), col("time")),
                "yyyy-MM-dd HH:mm:ss"
            )
        )
        )


In [0]:
races_selected_df = transformed_circuits_df.select(
    col("raceId"),
    col("year"),
    col("round"),
    col("circuitId"),
    col("name"),
    col("race_timestamp"),
    col("ingestion_date")
)


In [0]:
reces_renamed_df = (
    races_selected_df 
    .withColumnRenamed("raceId", "race_id")
    .withColumnRenamed("year","race_year")
    .withColumnRenamed("circuitId","circuit_id")
.withColumn("source", lit(v_source_value)))


### Step 3 - Write the output to parquet


In [0]:
processed_path = processed_folder_path


In [0]:
reces_renamed_df.write.mode("overwrite").partitionBy("race_year").parquet(f"{processed_path}/races")


In [0]:
spark.read.parquet(f"{processed_path}/races").display()


In [0]:
dbutils.notebook.exit("success")